In [1]:
# Завантаження GloVe
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip -d pretrained_models

--2025-02-22 21:09:47--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-02-22 21:09:47--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-02-22 21:09:47--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [2]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
!unzip wiki-news-300d-1M.vec.zip -d pretrained_models

--2025-02-22 21:12:46--  https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.165.160.120, 3.165.160.106, 3.165.160.70, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.165.160.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 681808098 (650M) [application/zip]
Saving to: ‘wiki-news-300d-1M.vec.zip’

wiki-news-300d-1M.v 100%[===================>] 650.22M   285MB/s    in 2.3s    

2025-02-22 21:12:48 (285 MB/s) - ‘wiki-news-300d-1M.vec.zip’ saved [681808098/681808098]

Archive:  wiki-news-300d-1M.vec.zip
  inflating: pretrained_models/wiki-news-300d-1M.vec  


In [3]:
import re
import os
import gc
import datetime
import time

import pandas as pd
import numpy as np

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import gensim
from gensim.models import word2vec
from gensim.models import KeyedVectors #  implements word vectors
from gensim.test.utils import datapath, get_tmpfile
from gensim.scripts.glove2word2vec import glove2word2vec

from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import matplotlib.cm as cm

import spacy

from wordcloud import WordCloud

from tqdm.auto import tqdm
tqdm.pandas()

import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("/kaggle/input/email-spam-detection-dataset-classification/spam.csv", encoding='ISO-8859-1')

In [5]:
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will Ì_ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [6]:
df.info()
df.isnull().sum()
print(df["v1"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB
v1
ham     4825
spam     747
Name: count, dtype: int64


Робимо первинний огляд. Бачимо що стовпці Unnamed: 2	Unnamed: 3	Unnamed: 4 містять багато порожніх данних. Приймаю рішення їх видалити. \
Бачимо чутливий бісбаланс ham та spam. Для того, щоб не спотворити результат зробивши штучны повыдомлення, обираю Oversampling\
Пізніший єксперімент надав читке розуміння що Oversampling не є гарним рішенням, та призводить до перенавчання. 

In [7]:
df_3columns = df.copy()
df_3columns.drop(columns = ['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], inplace = True)
df_3columns = df_3columns.rename(columns={"v1": "label", "v2": "message"})

# Перевіряємо баланс
print(df_3columns["label"].value_counts())

df_3columns['Spam'] = df_3columns['label'].apply(lambda x: 1 if x == "spam" else 0)

df_3columns

label
ham     4825
spam     747
Name: count, dtype: int64


,label,message,Spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will Ì_ b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


In [8]:
# Досліджуємо кількість дублікатів
df_3columns.duplicated().sum()

403

**Робимо підготовку тексту для навчання моделі**

In [9]:

# Contractions. Source http://stackoverflow.com/questions/19790188/expanding-english-language-contractions-in-python

contractions = {
"ain't": "am not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he would",
"he'd've": "he would have",
"he'll": "he will",
"he's": "he is",
"how'd": "how did",
"how'll": "how will",
"how's": "how is",
"i'd": "i would",
"i'll": "i will",
"i'm": "i am",
"i've": "i have",
"isn't": "is not",
"it'd": "it would",
"it'll": "it will",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"must've": "must have",
"mustn't": "must not",
"needn't": "need not",
"oughtn't": "ought not",
"shan't": "shall not",
"sha'n't": "shall not",
"she'd": "she would",
"she'll": "she will",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"that'd": "that would",
"that's": "that is",
"there'd": "there had",
"there's": "there is",
"they'd": "they would",
"they'll": "they will",
"they're": "they are",
"they've": "they have",
"wasn't": "was not",
"we'd": "we would",
"we'll": "we will",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"where'd": "where did",
"where's": "where is",
"who'll": "who will",
"who's": "who is",
"won't": "will not",
"wouldn't": "would not",
"you'd": "you would",
"you'll": "you will",
"you're": "you are"
}

In [10]:

# Stop-words
stop_words = set(stopwords.words('english')).union({'also', 'would', 'much', 'many'})

negations = {
    'aren',
    "aren't",
    'couldn',
    "couldn't",
    'didn',
    "didn't",
    'doesn',
    "doesn't",
    'don',
    "don't",
    'hadn',
    "hadn't",
    'hasn',
    "hasn't",
    'haven',
    "haven't",
    'isn',
    "isn't",
    'mightn',
    "mightn't",
    'mustn',
    "mustn't",
    'needn',
    "needn't",
    'no',
    'nor',
    'not',
    'shan',
    "shan't",
    'shouldn',
    "shouldn't",
    'wasn',
    "wasn't",
    'weren',
    "weren't",
    'won',
    "won't",
    'wouldn',
    "wouldn't"
}

stop_words = stop_words.difference(negations)

In [11]:
spacy.require_gpu() 
nlp = spacy.load("en_core_web_sm", disable = ['parser','ner'])

def normalize_text(raw_review):

    # Remove html tags
    text = re.sub("<[^>]*>", " ", raw_review) # match <> and everything in between. [^>] - match everything except >

    # Remove emails
    text = re.sub("\S*@\S*[\s]+", " ", text) # match non-whitespace characters, @ and a whitespaces in the end

    # remove links
    text = re.sub("https?:\/\/.*?[\s]+", " ", text) # match http, s - zero or once, //,
                                                    # any char 0-unlimited, whitespaces in the end

     # Convert to lower case, split into individual words
    text = text.lower().split()

    # Replace contractions with their full versions
    text = [contractions.get(word) if word in contractions else word
            for word in text]

    # Re-splitting for the correct stop-words extraction
    text = " ".join(text).split()

    # Remove stop words
    text = [word for word in text if not word in stop_words]

    text = " ".join(text)

    # Remove non-letters
    text = re.sub("[^a-zA-Z' ]", "", text) # match everything except letters and '


    # Stem words. Need to define porter stemmer above
    # text = [stemmer.stem(word) for word in text.split()]

    # Lemmatize words. Need to define lemmatizer above
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if len(token.lemma_) > 1 ])

    # Remove duplicate words
    words = text.split()
    text = " ".join(sorted(set(words), key=words.index))

    # Remove excesive whitespaces
    text = re.sub("[\s]+", " ", text)

    # Join the words back into one string separated by space, and return the result.
    return(text)

In [12]:
df_norm = df_3columns.copy()
df_norm['text_normalized'] = df_3columns['message'].progress_apply(normalize_text)

  0%|          | 0/5572 [00:00<?, ?it/s]

In [13]:
df_norm

,label,message,Spam,text_normalized
0,ham,"Go until jurong point, crazy.. Available only ...",0,go jurong point crazy available bugis great wo...
1,ham,Ok lar... Joking wif u oni...,0,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,0,dun say early hor already
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,nah not think go usf live around though
...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1,nd time try contact win pound prize claim easy...
5568,ham,Will Ì_ b going to esplanade fr home?,0,go esplanade fr home
5569,ham,"Pity, * was in mood for that. So...any other s...",0,pity mood that soany suggestion
5570,ham,The guy did some bitching but I acted like i'd...,0,guy bitching act like interested buy something...


In [14]:
def get_preds(text_column, algorithm, ngrams=(1,1)):

    start_time = time.time()  # Початок вимірювання часу

    X_train, X_test, y_train, y_test = train_test_split(df_norm[text_column], df_norm.label, test_size=0.2, stratify=df_norm['label'], random_state=42)
    
    if algorithm == 'cv':
        vect = CountVectorizer(ngram_range=ngrams).fit(X_train)
    elif algorithm == 'tfidf':
        vect = TfidfVectorizer(ngram_range=ngrams).fit(X_train)
    else:
        raise ValueError('Select correct algorithm: `cv` or `tfidf`')

    end_time = time.time()  # Кінець вимірювання часу
    elapsed_time = end_time - start_time  # Час виконання

    print('Vocabulary length: ', len(vect.vocabulary_))

    # transform the documents in the training data to a document-term matrix
    X_train_vectorized = vect.transform(X_train)
    print('Document-term matrix shape:', X_train_vectorized.shape)

    model = LogisticRegression(random_state=42)
    model.fit(X_train_vectorized, y_train)

    # Use predict_proba to get probabilities
    y_probs = model.predict_proba(vect.transform(X_test))[:, 1] 

    print(f'Used algorithm: {algorithm}')
    print('AUC: ', roc_auc_score(y_test, y_probs))  
    print('Accuracy: ', accuracy_score(y_test, model.predict(vect.transform(X_test))))
    print('F1-score: ', f1_score(y_test, model.predict(vect.transform(X_test)), pos_label='spam'))
    print(f'Time taken for {algorithm}: {elapsed_time:.4f} seconds')  # Вивести час виконання

In [15]:
get_preds('text_normalized','tfidf', (1,1))

Vocabulary length:  6544
Document-term matrix shape: (4457, 6544)
Used algorithm: tfidf
AUC:  0.9853196603999056
Accuracy:  0.9632286995515695
F1-score:  0.8416988416988417
Time taken for tfidf: 0.0589 seconds


In [16]:
get_preds('text_normalized','cv', (1,1))

Vocabulary length:  6544
Document-term matrix shape: (4457, 6544)
Used algorithm: cv
AUC:  0.985729570497589
Accuracy:  0.9811659192825112
F1-score:  0.9241877256317689
Time taken for cv: 0.0488 seconds


In [17]:
get_preds('message','cv', (1,1))

Vocabulary length:  7701
Document-term matrix shape: (4457, 7701)
Used algorithm: cv
AUC:  0.9857156752400406
Accuracy:  0.9811659192825112
F1-score:  0.9247311827956989
Time taken for cv: 0.0682 seconds


ДО Oversampling\
Спираючись на отримані показники можемо зробити вивод, що використання CountVectorizer, на нормалізованому тексті, демонструє кращі показники за TfidfVectorizer:\
час опрацювання: порівняний\ 
AUC: cv 0.9857, tfidf 0.9853\
Accuracy:  cv 0.9811, tfidf 0.9632\
F1-score:  cv 0.9241, tfidf 0.8416\
\
Якщо порівнювати нормалізований та ненормалізований тест, також бачимо значний приріст швидкості до 30% при нормалізації. 

Після застосування Oversampling модель демонструє надвиликі показники, що може свідчити про помилку або перенавчання

In [18]:
def build_corpus(data):
    "Creates a list of lists containing words from each sentence"
    corpus = []
    for sentence in data:
        word_list = sentence.split(" ")
        corpus.append(word_list)

    return corpus

In [19]:
corpus = build_corpus(df_norm['text_normalized'])
corpus[0]

['go',
 'jurong',
 'point',
 'crazy',
 'available',
 'bugis',
 'great',
 'world',
 'la',
 'buffet',
 'cine',
 'get',
 'amore',
 'wat']

Word Embeddings - методи представлення слів у вигляді векторів у багатовимірному просторі

In [20]:
os.makedirs('./saved_models', exist_ok=True)

In [21]:
# vector_size - Dimensionality of the word vectors
# window - Maximum distance between the current and predicted word within a sentence
# min_count - Ignores all words with total frequency lower than this

model_emb_from_scratch = word2vec.Word2Vec(corpus, vector_size=100, window=7, min_count=100, workers=0)

# saving vectors
model_emb_from_scratch.wv.save_word2vec_format('./saved_models/model_emb_from_scratch.bin', binary=True)

In [22]:
class WordEmbedding:

    def __init__(self):
        self.model = {}

    def convert(self, source, input_file_path, output_file_path):
        '''
        Converts word embeddings from GloVe format to Word2Vec format
        '''
        if source == 'glove':
            glove2word2vec(input_file_path, output_file_path)
        elif source in ['word2vec', 'fasttext', 'from_scratch']:
            pass
        else:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

    def load(self, source, file_path):
        '''
        Loads a specified word embedding model from a file
        '''
        print(datetime.datetime.now(), 'start: loading', source)
        if source in ['glove', 'fasttext']:
            self.model[source] = gensim.models.KeyedVectors.load_word2vec_format(file_path)
        elif source in ['word2vec', 'from_scratch']:
            self.model[source] = gensim.models.KeyedVectors.load_word2vec_format(file_path, binary=True)
        else:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        print(datetime.datetime.now(), 'end: loading', source)

        return self

    def get_model(self, source):
        '''
        Retrieves the loaded word embedding model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        return self.model[source]

    def get_words(self, source, size=None):
        '''
        Retrieves a list of words from the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        if size is None:
            return [w for w in self.get_model(source=source).key_to_index]
        else:
            results = []
            for i, word in enumerate(self.get_model(source=source).key_to_index):
                if i >= size:
                    break
                results.append(word)
            return results

        return Exception('Unexpected flow')

    def get_dimension(self, source):
        '''
        Retrieves the dimension of word vectors in the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        return self.get_model(source=source).vectors[0].shape[0]

    def get_vectors(self, source, words=None):
        '''
        Retrieves vectors for specified words or for all words in the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        if words is None:
            words = self.get_words(source=source)

        embedding = np.empty((len(words), self.get_dimension(source=source)), dtype=np.float32)
        for i, word in enumerate(words):
            embedding[i] = self.get_vector(source=source, word=word)

        return embedding

    def get_vector(self, source, word):
        '''
        Retrieves the vector representation of a single word
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        if source not in self.model:
            raise ValueError('Did not load %s model yet' % source)

        try:
            return self.model[source][word]
        except KeyError as e:
            dims = self.model[source][0].shape
            vect = np.empty(dims)
            vect[:] = np.nan
            return vect

    def get_synonym(self, source, word, topn=5):
        '''
        Retrieves synonyms for a given word
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        if source not in self.model:
            raise ValueError('Did not load %s model yet' % source)

        try:
            return self.model[source].most_similar(positive=word, topn=topn)
        except KeyError as e:
            raise

    def get_distance_between_two_words(self, source, word1, word2):
        '''
        Calculates cosine similarity between two words in the model
        '''
        if source not in ['glove', 'word2vec', 'fasttext', 'from_scratch']:
            raise ValueError('Possible value of source are glove, word2vec, fasttext, or from_scratch')

        if source not in self.model:
            raise ValueError('Did not load %s model yet' % source)

        try:
            return self.model[source].similarity(word1, word2)
        except KeyError as e:
            raise


In [23]:
os.makedirs('./pretrained_models', exist_ok=True)
os.makedirs('./saved_models', exist_ok=True)
print(os.listdir('./pretrained_models'))
print(os.listdir('./saved_models'))

['glove.6B.50d.txt', 'wiki-news-300d-1M.vec', 'glove.6B.300d.txt', 'glove.6B.200d.txt', 'glove.6B.100d.txt']
['model_emb_from_scratch.bin']


Для завантаження авантаження Google News Word2Vec:\
Перейдіть за посиланням і завантажте файл: 🔗 GoogleNews-vectors-negative300.bin\
Або використайте альтернативне посилання: 🔗 GoogleNews-vectors-negative300.bin.gz (Google Drive)\
Помістіть його у папку pretrained_models/

In [24]:
# Різноманіття текстових-у-вектор моделей
# word2vec_file_path = './pretrained_models/GoogleNews-vectors-negative300.bin' потребує додаткового завантаження

from_scratch_file_path = './saved_models/model_emb_from_scratch.bin'

fasttext_file_path = './pretrained_models/wiki-news-300d-1M.vec'

# adding absolute path for correct gensim work
downloaded_glove_file_path = './pretrained_models' + '/glove.6B.50d.txt'
glove_file_path = './pretrained_models' + '/glove.6B.50d.vec'

In [25]:
word_embedding = WordEmbedding()

In [26]:
print("downloaded_glove_file_path:", downloaded_glove_file_path)
print("glove_file_path:", glove_file_path)

downloaded_glove_file_path: ./pretrained_models/glove.6B.50d.txt
glove_file_path: ./pretrained_models/glove.6B.50d.vec


In [27]:
# Завантажуємо GloVe з вебсайту та конвертуємо цільовий файл у векторний формат
word_embedding.convert('glove', downloaded_glove_file_path, glove_file_path)

<ipython-input-22-ae5082a751a6>:11: DeprecationWarning: Call to deprecated `glove2word2vec` (KeyedVectors.load_word2vec_format(.., binary=False, no_header=True) loads GLoVE text vectors.).
  glove2word2vec(input_file_path, output_file_path)


In [28]:
word_embedding.load('glove', glove_file_path)
word_embedding.load('fasttext', fasttext_file_path)
word_embedding.load(source='from_scratch', file_path=from_scratch_file_path)

# word_embedding.load(source='word2vec', file_path=word2vec_file_path)

2025-02-22 21:14:47.139800 start: loading glove
2025-02-22 21:14:59.821828 end: loading glove
2025-02-22 21:14:59.822147 start: loading fasttext
2025-02-22 21:17:35.687710 end: loading fasttext
2025-02-22 21:17:35.688018 start: loading from_scratch
2025-02-22 21:17:35.689706 end: loading from_scratch


In [29]:
def tok2vec(tokens, source:str, avg:str):
    """
    Given a list of tokens, return their vector representation.
    Args:
        tokens: List(str) tokenized input
        source: embedding algorithm to use with the WordEmbedding object
        avg: vectors averaging method - `sum` or `mean` of all vectors
    """
    vects = word_embedding.get_vectors(source=source, words=tokens)
    expected_vector_size = word_embedding.get_dimension(source=source)
    if len(vects) == 0 or np.all(np.isnan(vects)):
        return np.zeros(expected_vector_size)

    if avg == 'mean':
        return np.nanmean(vects, axis=0)
    elif avg == 'sum':
        return np.nansum(vects, axis=0)
    else:
        raise ValueError('Select correct averaging method: sum or mean')

In [30]:
def get_preds_with_embeddings(df_norm, text_column, source:str, avg='sum'):

    start_time = time.time()  # Початок вимірювання часу
    
    X_train, X_test, y_train, y_test = train_test_split(df_norm[text_column],
                                                        df_norm.Spam,
                                                        test_size=0.2,
                                                        stratify=df_norm['Spam'], random_state=42)
    X_train = X_train.apply(word_tokenize).apply(lambda x: tok2vec(x, source, avg)).to_numpy()
    X_test = X_test.apply(word_tokenize).apply(lambda x: tok2vec(x, source, avg)).to_numpy()

    X_train = np.stack(X_train, axis=0)
    X_test = np.stack(X_test, axis=0)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    end_time = time.time()  # Кінець вимірювання часу
    execution_time = end_time - start_time
     
    print(f'Used algorithm: {source}')
    print('AUC: ', roc_auc_score(y_test, predictions))
    print('Accuracy: ', accuracy_score(y_test, predictions))
    print('F1-score: ', f1_score(y_test, predictions))
    print(classification_report(y_test, predictions))
    print(f"Execution Time: {execution_time:.4f} seconds")

Метрики AUC, Accuracy та F1-score були обрані як сталі для ML та DL навчання.\
Вони є зрозумілі та наочні.\

Метрика Time використана як додаткова для прийняття рішення, з єкономічної точки зору. Вона дозволяє порівняти час на обробку даних , та відсоток вірних припущень   

In [31]:
get_preds_with_embeddings(df_norm, 'text_normalized', source='glove')

Used algorithm: glove
AUC:  0.8763391554462462
Accuracy:  0.957847533632287
F1-score:  0.8290909090909092
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       966
           1       0.90      0.77      0.83       149

    accuracy                           0.96      1115
   macro avg       0.93      0.88      0.90      1115
weighted avg       0.96      0.96      0.96      1115

Execution Time: 0.8400 seconds


In [32]:
get_preds_with_embeddings(df_norm, 'text_normalized', source='fasttext')

Used algorithm: fasttext
AUC:  0.9493969458223908
Accuracy:  0.9811659192825112
F1-score:  0.9278350515463918
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.95      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Execution Time: 0.8257 seconds


In [33]:
get_preds_with_embeddings(df_norm, 'text_normalized', source='from_scratch')

Used algorithm: from_scratch
AUC:  0.5
Accuracy:  0.8663677130044843
F1-score:  0.0
              precision    recall  f1-score   support

           0       0.87      1.00      0.93       966
           1       0.00      0.00      0.00       149

    accuracy                           0.87      1115
   macro avg       0.43      0.50      0.46      1115
weighted avg       0.75      0.87      0.80      1115

Execution Time: 0.7359 seconds


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [34]:
get_preds_with_embeddings(df_norm, 'message', source='fasttext')

Used algorithm: fasttext
AUC:  0.952235052176692
Accuracy:  0.9811659192825112
F1-score:  0.9283276450511946
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.94      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115

Execution Time: 1.5649 seconds


Найкращі результати демонструє Fasttext. Він повільніший за Glove, але демонструє кращі показники особливо це важливо зважаючи на те, що він краще відрізняє Спам, що є цільовим, для цього алгоритму
\
Після застосуання Oversampling показники досягли надмірних показників 



In [35]:
# get_preds_with_embeddings(df_norm, 'text_normalized', source='word2vec')